# QLoRA 실습
## 실습 개요
본 실습에서는 QLoRA(Quantized Low-Rank Adaptation) 기법을 사용하여 거대 언어 모델을 적은 자원으로 효율적으로 파인튜닝하는 과정을 학습합니다.
## 실습 목표
- QLoRA의 핵심 개념(4-bit quantization, LoRA adapters)을 이해한다.
- `bitsandbytes`, `peft` 라이브러리를 활용하여 양자화된 모델을 로드하고 학습 준비를 할 수 있다.
- 사용자 정의 데이터셋을 로드하고 Chat Format에 맞게 전처리할 수 있다.
- `Trainer`를 사용하여 학습을 수행하고, 학습된 어댑터를 저장할 수 있다.
## Prerequisites
- **Python**: 3.8 이상
- **Library**: bitsandbytes, transformers, peft, accelerate, datasets, wandb
- **Hardware**: GPU
## 데이터셋 정보
- **Dataset Name**: DopeorNope/Ko-Optimize_Dataset
- **License**: (데이터셋의 라이선스 확인 필요, 통상적으로 CC-BY-SA 등을 따름)
- **Description**: 한국어 지시 이행(Instruction Following) 데이터셋

In [1]:
# !pip install bitsandbytes
# !pip install datasets
# !pip install -q -U git+https://github.com/huggingface/transformers.git
# !pip install -q -U git+https://github.com/huggingface/peft.git
# !pip install -q -U git+https://github.com/huggingface/accelerate.git
# !pip install pandas
# !pip install wandb
# !pip install lomo-optim

# 토크나이저 및 데이터 준비

## Text Data 준비

In [2]:
from datasets import load_dataset
import pandas as pd
from huggingface_hub import login

/home/xeong_oxx/miniconda3/envs/qlora_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# 허깅페이스 로그인 방법
my_hf_key=''
login(my_hf_key)

In [4]:
# 모델 레포지토리
model_path = "ryusangwon/ko_en_Llama-3.2-1B-Instruct"

# 데이터 path
data_path = 'DopeorNope/Ko-Optimize_Dataset'

In [5]:
# dataset 다운

data = load_dataset(data_path)

In [6]:
data

DatasetDict({
    train: Dataset({
        features: ['input', 'instruction', 'output'],
        num_rows: 10000
    })
})

In [7]:
data['train']

Dataset({
    features: ['input', 'instruction', 'output'],
    num_rows: 10000
})

In [8]:
df= pd.DataFrame(data)

In [9]:
df.head()

,train
0,"{'input': '', 'instruction': '∫[0,t] (GCP × SM..."
1,{'input': '귀하는 사람들이 정보를 찾도록 도와주는 AI 어시스턴트입니다. ...
2,"{'input': '', 'instruction': '이산화탄소 배출을 줄이는 5가..."
3,{'input': '귀하는 사람들이 정보를 찾도록 도와주는 AI 어시스턴트입니다. ...
4,{'input': '귀하는 항상 설명을 제공하는 도움이 되는 조수입니다. 5살짜리 ...


In [10]:
df = pd.DataFrame(data['train'])

In [11]:
# 데이터셋 구성확인
df.head()

,input,instruction,output
0,,"∫[0,t] (GCP × SM × TA) dt = SA\n\n여기서\n∫ = 적분 ...","예, 주어진 공식을 다음과 같이 수정할 수 있습니다:\n\n∫[0,T] [(GCP ..."
1,귀하는 사람들이 정보를 찾도록 도와주는 AI 어시스턴트입니다. 사용자가 질문을 합니...,정답 -> 문장 A\n질문 -> 다음 두 문장 중 상식에 어긋나는 것은 어느 것입니...,질문 -> 다음 두 문장 중 상식에 어긋나는 것은 어느 것입니까?\n옵션:\n- 문...
2,,이산화탄소 배출을 줄이는 5가지 방법을 나열하세요.,"1. 재생 에너지원 사용: 태양열, 풍력, 수력, 지열 등 이산화탄소를 배출하지 않..."
3,귀하는 사람들이 정보를 찾도록 도와주는 AI 어시스턴트입니다. 사용자가 질문을 합니...,"많은 정치인들이 소농에 대해 이야기하는 것을 좋아하지만, 실제로 거의 모든 농장은 ...",서서히 해봅시다: 노화된 록스타의 손이 협조하지 않는 것은 건강 문제를 나타냅니다....
4,귀하는 항상 설명을 제공하는 도움이 되는 조수입니다. 5살짜리 아이에게 대답한다고 ...,다음 리뷰의 감상을 알려주세요: 솔직히 저는 사춘기 욕구 때문에 이 영화를 봤어요....,이 리뷰에는 긍정적인 감정이 담겨 있습니다. 영화를 보고 유머와 연기를 즐기며 즐거...


In [12]:
import datasets

In [13]:
# 기존에 pandas df를 hf의 모델 학습에 맞게 변환해줌 / split 나누기
test=datasets.DatasetDict({
    'train':datasets.Dataset.from_pandas(df),
    'test':datasets.Dataset.from_pandas(df)
})

In [14]:
test.push_to_hub('Jeongseongwoo08/upstage_lora_finetuning_practice',private=False, token=my_hf_key)

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 10.65ba/s]
Processing Files (1 / 1): 100%|██████████| 7.54MB / 7.54MB,  0.00B/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 10.80ba/s]
Processing Files (1 / 1): 100%|██████████| 7.54MB / 7.54MB,  0.00B/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.04s/ shards]
No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/Jeongseongwoo08/upstage_lora_finetuning_practice/commit/92a4c502d8d211aa9e71660f45bf3e2007bf3b28', commit_message='Upload dataset', commit_description='', oid='92a4c502d8d211aa9e71660f45bf3e2007bf3b28', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Jeongseongwoo08/upstage_lora_finetuning_practice', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Jeongseongwoo08/upstage_lora_finetuning_practice'), pr_revision=None, pr_num=None)

In [15]:
import os
import ctypes
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"
# 1. 꼬여있는 라이브러리 경로 직접 지정
lib_path = "/home/xeong_oxx/miniconda3/envs/py11/lib/libmkl_rt.so"

# 2. torch 임포트 전 라이브러리를 전역 모드로 로드
try:
    ctypes.CDLL(lib_path, mode=ctypes.RTLD_GLOBAL)
    print("MKL 라이브러리 로드 성공")
except Exception as e:
    print(f"로드 실패: {e}")

# 3. 그 다음 torch 임포트
import torch
print(f"PyTorch 로드 완료: {torch.cuda.is_available()}")
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments, DataCollatorForSeq2Seq

import bitsandbytes as bnb

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training)


MKL 라이브러리 로드 성공
PyTorch 로드 완료: True


KeyboardInterrupt: 

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_path)

In [ ]:
# 문장이 끝나는 지점 -> 이 토큰 내뱉으면 문장 끝임
tokenizer.eos_token_id

128009

In [ ]:
tokenizer.bos_token_id

128000

In [ ]:
tokenizer.decode([128009])

'<|eot_id|>'

In [ ]:
# 토크나이저 세팅: QLoRA시 pad 토큰을 eos로 설정해주기
# 문장의 길이가 다름 => 문장의 길이가 동일하게 맞춰줌 - 패딩처리
bos = tokenizer.bos_token_id # begin of sentence
eos = tokenizer.eos_token_id # end of sentence

tokenizer.add_special_tokens({"pad_token": "<|reserved_special_token_0|>"})

0

In [ ]:
tokenizer.padding_side = 'right' # padding to right (otherwise SFTTrainer shows warning)

#추론할 때는 다르겠죠

In [ ]:
pad = tokenizer.pad_token_id

# tokenizer.pad_token_id = eos

cut_off_len = 4098
val_size = 0.005
train_on_inputs = False #<질문><답변> -> Loss를 안줌, 다만 train on inputs 옵션을 주면 질문에도 loss를 줄 수 잇음

add_eos_token = False

In [ ]:
template = {
    "prompt_input": "아래는 문제를 설명하는 지시사항과, 구체적인 답변을 방식을 요구하는 입력이 함께 있는 문장입니다. 이 요청에 대해 적절하게 답변해주세요.\n###입력:{input}\n###지시사항:\n{instruction}\n###답변:\n",
    "prompt_no_input": "아래는 문제를 설명하는 지시사항입니다. 이 요청에 대해 적절하게 답변해주세요.\n###지시사항:\n{instruction}\n###답변:\n"
}

In [ ]:
from typing import Union

def generate_prompt(
    instruction: str,
    input: Union[None, str] = None,
    label: Union[None, str] = None,
    verbose: bool = False
) -> str:
    """
    주어진 instruction, input, label을 사용하여 프롬프트를 생성하는 함수.

    Parameters:
    - instruction (str): 문제 설명 또는 지시사항.
    - template (dict): 입력이 있는 경우와 없는 경우의 템플릿을 포함한 딕셔너리.
    - input (str or None): 문제에 대한 구체적인 입력 (옵션).
    - label (str or None): 정답 또는 응답 (옵션).
    - verbose (bool): 생성된 프롬프트를 출력할지 여부.

    Returns:
    - str: 완성된 프롬프트.
    """
    if input:
        # input이 있으면 prompt_input에 format을 이용해서 instruction이랑 input 인자 전달
        res = template["prompt_input"].format(instruction=instruction, input=input)
    else:
        res = template["prompt_no_input"].format(instruction=instruction)

    if label:
        res = f"{res}{label}"

    if verbose:
        print(res)

    return res

In [ ]:
def tokenize(prompt, add_eos_token=True):
   result = tokenizer(prompt,truncation=True,max_length=cut_off_len,padding=False,return_tensors=None,)
   if (result["input_ids"][-1] != tokenizer.eos_token_id
       and len(result["input_ids"]) < cut_off_len
       and add_eos_token
      ):
        result["input_ids"].append(tokenizer.eos_token_id)
        result["attention_mask"].append(1)

   result["labels"] = result["input_ids"].copy()
   return result

In [ ]:
def generate_and_tokenize_prompt(data_point):
    full_prompt = generate_prompt(
        data_point["instruction"],
        data_point["input"],
        data_point["output"]
        )
    tokenized_full_prompt = tokenize(full_prompt)
    if not train_on_inputs:
        user_prompt = generate_prompt(data_point["instruction"], data_point["input"])
        tokenized_user_prompt = tokenize(user_prompt, add_eos_token=add_eos_token)
        user_prompt_len = len(tokenized_user_prompt["input_ids"])

        if add_eos_token:
            user_prompt_len -= 1



        tokenized_full_prompt["labels"] = [-100] * user_prompt_len + tokenized_full_prompt["labels"][user_prompt_len:]

    return tokenized_full_prompt

In [ ]:
# 제거할 문자열 컬럼 리스트 (실제 데이터셋의 컬럼명에 맞춰 조정하세요)
# 보통 'instruction', 'input', 'output', 'status' 등이 해당됩니다.
remove_columns = ["instruction", "input", "output"] 

if val_size > 0:
    train_val = data["train"].train_test_split(test_size=val_size, shuffle=True, seed=42)
    train_data = train_val["train"].shuffle().map(
        generate_and_tokenize_prompt, 
        remove_columns=train_val["train"].column_names # 기존 모든 컬럼을 지우고 토큰화된 결과만 남김
    )
    val_data = train_val["test"].shuffle().map(
        generate_and_tokenize_prompt,
        remove_columns=train_val["test"].column_names
    )
else:
    train_data = data["train"].shuffle().map(
        generate_and_tokenize_prompt,
        remove_columns=data["train"].column_names
    )
    val_data = None

Map: 100%|██████████| 50/50 [00:00<00:00, 1229.89 examples/s]


In [ ]:
print(train_data['labels'][0])

[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -10

In [ ]:
a=[ 40, 13, 125910, 122691, 102080, 18359, 105519, 44690, 61938, 382, 102546, 80732, 512, 13094, 122795, 101106, 77437, 107651, 57575, 107120, 110709, 29102, 103684, 106788, 117266, 16969, 105519, 44690, 81673, 106434, 53400, 105512, 32179, 90759, 109670, 13, 39623, 113, 93131, 602, 20565, 107120, 110709, 29102, 103684, 111436, 16969, 75908, 101136, 103236, 101563, 104, 18359, 105519, 44690, 24486, 106788, 125910, 45618, 29726, 105519, 123935, 111809, 83290, 105519, 44690, 121002, 21028, 67236, 61394, 102199, 102914, 18918, 124208, 67525, 106117, 80052, 382, 36092, 113, 93131, 14799, 16969, 103236, 101563, 104, 18359, 102293, 104381, 226, 107496, 58901, 55000, 21121, 106958, 16633, 236, 56154, 113047, 112822, 116942, 106647, 41820, 44005, 110005, 106751, 82068, 13094, 109745, 102326, 27797, 103684, 107316, 13094, 104231, 101203, 90195, 45780, 242, 120, 101003, 71023, 54780, 106434, 13094, 47782, 34609, 117622, 104350, 101974, 100968, 22035, 51796, 39331, 382, 36092, 113, 93131, 220, 126892, 34804, 91767, 103236, 101563, 104, 105519, 123935, 107762, 34609, 117622, 102484, 34804, 105825, 53987, 102, 18359, 104423, 20565, 105316, 43139, 111335, 119, 18359, 108289, 20565, 47782, 34609, 117622, 104350, 101974, 100968, 22035, 51796, 39331, 382, 36092, 113, 93131, 220, 49412, 96, 16969, 101480, 21028, 57139, 108859, 103236, 101563, 104, 105519, 44690, 21028, 124191, 105316, 54780, 106434, 13094, 120078, 13, 128009]

In [ ]:
print(tokenizer.decode(a))

I. 자동차 좌석을 청소합니다.

설명:
이 객관식 문제에서 가장 논리적인 다음 이벤트는 청소와 관련된 것이어야 합니다. 옵션 i가 가장 논리적인 이유는 방금 카펫을 청소한 다음 자동차 시트 청소를 진행하여 청소 작업의 내러티브를 유지하기 때문입니다.

옵션 ii는 카펫을 매끄럽게 하기 위해 잎사귀 송풍기를 사용하는 것은 이상적이거나 실용적인 방법이 아니며 커피 유출과 관련이 없으므로 올바르지 않습니다.

옵션 ③은 이미 카펫 청소를 했으므로 남은 얼룩을 손가락으로 씻을 필요가 없으므로 올바르지 않습니다.

옵션 ④는 무의미하며 카펫 청소의 맥락과 관련이 없습니다.<|eot_id|>


In [ ]:
print(tokenizer.decode(train_data['input_ids'][0]))

<|begin_of_text|>아래는 문제를 설명하는 지시사항입니다. 이 요청에 대해 적절하게 답변해주세요.
###지시사항:
주가에 대한 2주기 이항 모형의 경우 다음이 주어집니다. (i) 각 기간은 6개월입니다. (ii) 무배당 주식의 현재 가격은 $70.00입니다. (iii) u = 1.181이며, 여기서 u는 주가가 상승할 경우 기간당 주식의 자본 이득률에 1을 더한 값입니다. (iv) d = 0.890, 여기서 d는 1에 가격이 하락할 경우 기간당 주식에 대한 자본 손실률을 더한 값입니다. (v) 연속 복리 무위험 이자율은 5%입니다. 행사 가격이 $80.00인 주식에 대한 1년 미국 풋 옵션의 현재 가격은 얼마입니까?
관련 정리: 이항 모형은 옵션 및 기타 금융 파생상품의 가치를 평가하기 위해 금융에서 사용되는 정량적 방법입니다. 특정 기간 동안 기초 자산의 가능한 가격 변동을 나타내는 이산 시간 모델입니다. 이 모델은 자산 가격이 각 시간 단계에서 특정 비율만큼만 상승 또는 하락할 수 있다는 가정을 기반으로 하며, 가능한 가격 경로의 이항 트리를 생성합니다.

이항 모형의 주요 구성 요소는 다음과 같습니다:

1. 시간 단계: 이 모델은 옵션 만기까지의 시간을 시간 단계라고 하는 일련의 동일한 간격으로 나눕니다. 각 시간 단계는 기초자산 가격이 변할 수 있는 가능한 시점을 나타냅니다.

2. 위아래 움직임: 각 시간대에서 기초자산 가격은 일정 계수(u)만큼 상승하거나 일정 계수(d)만큼 하락할 수 있습니다. 이러한 계수는 일반적으로 자산의 변동성과 시간 단계의 길이에 따라 결정됩니다.

3. 확률: 모델은 각 시간 단계의 상승 및 하락 움직임에 확률을 할당합니다. 이러한 확률은 일반적으로 무위험 이자율과 자산의 기대 수익률을 사용하여 계산되는 위험 중립 확률을 기반으로 합니다.

4. 수익: 이 모델은 만기일에 가능한 각 가격 경로에서 옵션의 수익을 계산합니다. 콜옵션의 경우 자산 가격이 행사가격보다 높으면 자산 가격과 행사가격의 차액이, 그렇지 않

In [ ]:
len(train_data['input_ids'][0])

775

In [ ]:
a="""아래는 문제를 설명하는 지시사항과, 구체적인 답변을 방식을 요구하는 입력이 함께 있는 문장입니다. 이 요청에 대해 적절하게 답변해주세요.\n###입력:{input}\n###지시사항:{instruction}\n###답변\n"""

# Model 준비

In [ ]:
# Quantization config 준비 -> FFT 시는 제외
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_storage=torch.float16,)





In [ ]:
# Model 로드 하기
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config = quantization_config, #-> FFT 시는 제외
    torch_dtype = torch.bfloat16,
    device_map = "auto",
    trust_remote_code = True
    )


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 146/146 [00:00<00:00, 250.87it/s, Materializing param=model.norm.weight]                              


In [ ]:
model.config.pad_token_id = tokenizer.pad_token_id # eos 토큰을 pad 토큰으로 쓸 경우는 제외하세요!

In [ ]:

# 우리가 모델을 불러왔는데 -> 학습불가능한 형태로 freeze
model = prepare_model_for_kbit_training(model)

In [ ]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm

In [ ]:
config = LoraConfig(
    r = 16,
    lora_alpha = 32,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout = 0.05,
    bias = "none",
    task_type = "CAUSAL_LM"
    )

In [ ]:
def find_all_linear_names(model):
  cls = bnb.nn.Linear4bit
  lora_module_names = set()
  for name, module in model.named_modules():
    if isinstance(module, cls):
      names = name.split('.')
      lora_module_names.add(names[0] if len(names) == 1 else names[-1])
  return list(lora_module_names)

In [ ]:
print('Trainable targer module:',find_all_linear_names(model))

Trainable targer module: ['down_proj', 'v_proj', 'o_proj', 'up_proj', 'q_proj', 'gate_proj', 'k_proj']


In [ ]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm

In [ ]:
# QLoRA 준비 -> FFT 시는 제외
model = get_peft_model(model, config)

In [ ]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [ ]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

In [ ]:
# 파라미터 수 체크
print_trainable_parameters(model)

trainable params: 3407872 || all params: 509413376 || trainable%: 0.6689796853704917


In [ ]:
[1,2,3,4],[1,2,3,4],[1,2,3,4],[1,2,3,4],[1,2,3,4],[1,2,3,4],[1,2,3,4],[1,2,3,4],[1,2,3,4],[1,2,3,4]

([1, 2, 3, 4],
 [1, 2, 3, 4],
 [1, 2, 3, 4],
 [1, 2, 3, 4],
 [1, 2, 3, 4],
 [1, 2, 3, 4],
 [1, 2, 3, 4],
 [1, 2, 3, 4],
 [1, 2, 3, 4],
 [1, 2, 3, 4])

In [ ]:
#Hyper parameter setting
output_dir='./llama_singleGPU-v1'
[1,2,3,4,5,6,7,8,9]

num_epochs = 1
micro_batch_size = 1
gradient_accumulation_steps = 128
warmup_steps = 10
learning_rate = 5e-7

group_by_length = False



#optimizer =  'adalomo'
optimizer ='paged_adamw_8bit'

# adam 활용시
beta1 = 0.9
beta2 = 0.99

lr_scheduler = 'cosine'
logging_steps = 1

use_wandb = True
wandb_run_name = 'Single_GPU_Optim'

use_fp16 = False
use_bf_16 = True
evaluation_strategy = 'steps'
eval_steps = 50
save_steps = 50
save_strategy = 'steps'




In [ ]:
model.gradient_checkpointing_enable()

In [ ]:
from transformers import Trainer

# Trainer를 상속받아 문제의 구간만 수정합니다.
class NoParallelTrainer(Trainer):
    def _wrap_model(self, model, training=True, dataloader=None):
        # Trainer가 모델을 DataParallel로 감싸려 할 때, 
        # 이를 무시하고 원래 모델(device_map이 적용된 상태)을 그대로 반환합니다.
        return model
model.is_parallelizable = True
model.model_parallel = True
trainer = NoParallelTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=val_data,
    args=TrainingArguments(
    per_device_train_batch_size = micro_batch_size,
    per_device_eval_batch_size = micro_batch_size,
    gradient_accumulation_steps = gradient_accumulation_steps,
    warmup_steps = warmup_steps,
    num_train_epochs = num_epochs,
    learning_rate = learning_rate,
    adam_beta1 = beta1, # adam 활용할때 사용
    adam_beta2 = beta2, # adam 활용할때 사용
    fp16 = use_fp16,
    bf16 = use_bf_16,
    gradient_checkpointing=True,
    #WandB 설정 (RANK 0 에서만 실행되도록 Trainer가 자동 처리)
    report_to="wandb" if use_wandb else "none",
    run_name=wandb_run_name if use_wandb else None,
    logging_steps = logging_steps,
    optim = optimizer,
    eval_strategy = evaluation_strategy if val_size > 0 else "no",
    save_strategy="steps",  #스텝기준으로 save
    eval_steps = eval_steps if val_size > 0 else None,
    save_steps = save_steps,
    lr_scheduler_type=lr_scheduler,
    output_dir = output_dir,
    load_best_model_at_end = True if val_size > 0 else False ,
    group_by_length=group_by_length,
    ddp_find_unused_parameters=False,
    remove_unused_columns=False
    ), 
    data_collator=DataCollatorForSeq2Seq(
        tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True
        ),
    )

In [ ]:
model.config.use_cache = False


trainer.train()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/xeong_oxx/.netrc.
wandb: Currently logged in as: mytnt831 (mytnt831-kangwon-national-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss


TrainOutput(global_step=20, training_loss=45.66546268463135, metrics={'train_runtime': 528.5439, 'train_samples_per_second': 18.825, 'train_steps_per_second': 0.038, 'total_flos': 4.454004909028147e+16, 'train_loss': 45.66546268463135, 'epoch': 1.0})

In [ ]:
trainer.save_model()
tokenizer.save_pretrained(output_dir)


('./llama_singleGPU-v1/tokenizer_config.json',
 './llama_singleGPU-v1/chat_template.jinja',
 './llama_singleGPU-v1/tokenizer.json')

In [ ]:
from peft import PeftModel


base_model= AutoModelForCausalLM.from_pretrained(model_path, token= my_hf_key)


merged_model= PeftModel.from_pretrained(base_model, output_dir)
merged_model= merged_model.merge_and_unload()

merged_model.push_to_hub('your_repo_name')
tokenizer.push_to_hub('your_repo_name')

Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.06s/it]
Processing Files (1 / 1): 100%|██████████| 4.94GB / 4.94GB, 8.83MB/s  
New Data Upload: 100%|██████████|  673MB /  673MB, 8.83MB/s  
Processing Files (1 / 1): 100%|██████████| 17.2MB / 17.2MB, 12.3MB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  


CommitInfo(commit_url='https://huggingface.co/Jeongseongwoo08/your_repo_name/commit/581352db3ec743ab21c58f38c9f3c53c8f3a7208', commit_message='Upload tokenizer', commit_description='', oid='581352db3ec743ab21c58f38c9f3c53c8f3a7208', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Jeongseongwoo08/your_repo_name', endpoint='https://huggingface.co', repo_type='model', repo_id='Jeongseongwoo08/your_repo_name'), pr_revision=None, pr_num=None)

: 

###**콘텐츠 라이선스**
<font color='red'><b>**(주)업스테이지가 제공하는 모든 교육 콘텐츠의 지식재산권은
운영 주체인 (주)업스테이지 또는 해당 저작물의 적법한 관리자에게 귀속되어 있습니다.**</b></font>

콘텐츠 일부 또는 전부를 **복사, 복제, 판매, 재판매 공개, 공유** 등을 할 수 없습니다. 유출될 경우 지식재산권 침해에 대한 책임을 부담할 수 있습니다.

유출에 해당하여 금지되는 행위의 예시는 다음과 같습니다.
* 콘텐츠를 재가공하여 온/오프라인으로 공개하는 행위
* 콘텐츠의 일부 또는 전부를 이용하여 인쇄물을 만드는 행위
* 콘텐츠의 전부 또는 일부를 녹취 또는 녹화하거나 녹취록을 작성하는 행위
* 콘텐츠의 전부 또는 일부를 스크린 캡쳐하거나 카메라로 촬영하는 행위
* 지인을 포함한 제3자에게 콘텐츠의 일부 또는 전부를 공유하는 행위
* 다른 정보와 결합하여 Upstage Education의 콘텐츠임을 알아볼 수 있는 저작물을 작성, 공개하는 행위
* 제공된 데이터의 일부 혹은 전부를 Upstage Education 프로젝트/실습 수행 이외의 목적으로 사용하는 행위